In [40]:
pip install --upgrade langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [41]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Ensure the Google API key is present
if "GOOGLE_API_KEY" not in os.environ:
    raise ValueError("GOOGLE_API_KEY is not set in the .env file")

In [42]:
import json

# Read the file "gods.json" and load it into the variable "gods"
# Note: Ensure the path is correct based on your project structure (e.g., 'data/gods.json')
with open("data/gods.json", "r", encoding="utf-8") as file:
    gods = json.load(file)

print('Total number of items:', len(gods))

Total number of items: 14


In [43]:



# Convert the gods list into a well-formatted JSON string for the prompt context
gods_context = json.dumps(gods, ensure_ascii=False, indent=2)
gods_context = gods_context.replace("{", "{{").replace("}", "}}")

# Create a robust system message with strict LLM guidelines for the autograder
system_message = f"""You are an expert assistant on Greek Mythology.
Use ONLY the following context to answer the user's questions.

Context:
{gods_context}

CRITICAL INSTRUCTIONS:
1. Output ONLY the raw keywords.
2. NEVER use full sentences, verbs, or prepositions (like در, از, به).
3. NEVER use the conjunction "و" (and). Use commas instead to separate items.
4. Maximum limit: 5 words.
5. If the answer is not explicitly available in the context, you MUST reply exactly with "نمی‌دانم" and nothing else.
6. Always respond in Persian (Farsi).
"""

# Make a list of at least 10 high-quality examples with unique phrasing
examples = [
    {"question": "کدام الهه با آتشدان و کانون خانواده مرتبط است؟", "answer": "هستیا"},
    {"question": "ارباب دنیای زیرین و فرمانروای مردگان چه کسی است؟", "answer": "هادس"},
    {"question": "خدای آهنگری، فلزکاری و آتش چه نام دارد؟", "answer": "هفائستوس"},
    {"question": "نام الهه مرتبط با خرد و استراتژی جنگی چیست؟", "answer": "آتنا"},
    {"question": "چه کسی به عنوان پیام‌رسان سریع خدایان المپ شناخته می‌شود؟", "answer": "هرمس"},
    {"question": "خدای حامی موسیقی، شعر و نور چه نام دارد؟", "answer": "آپولون"},
    {"question": "کدام الهه نماد کشاورزی، غلات و برداشت محصول است؟", "answer": "دمتر"},
    {"question": "الهه زیبایی و عشق که از امواج دریا زاده شد کیست؟", "answer": "آفرودیت"},
    {"question": "خدای مرتبط با شراب، رویش انگور و جشن‌ها چه کسی است؟", "answer": "دیونیسوس"},
    {"question": "کدام الهه شکارچی، خواهر دوقلوی آپولون محسوب می‌شود؟", "answer": "آرتمیس"}
]

In [44]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Define the template for formatting a single example
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)

# Create the few-shot prompt combining all examples
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Create the final prompt by combining the system message, few-shot prompt, and user query
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        few_shot_prompt,
        ("human", "{question}"),
    ]
)

In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# Initialize the Gemini model with temperature 0 for deterministic outputs
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0,
    max_tokens=15 # Hard limit to prevent verbosity
)

# Make a chain with the prompt, model, and string output parser
chain = prompt | model | StrOutputParser()

In [ ]:
import pandas as pd
import time
import re

def clean_response(text):
    # Strip whitespace
    text = text.strip()
    
    # Replace the Farsi conjunction "و" with a comma to prevent word count issues
    text = text.replace(" و ", "، ")
    
    # Fix zero-width non-joiner for the default unknown response
    if text in ["نمی دانم", "نمیدانم"]:
        text = "نمی‌دانم"
        
    # Remove trailing periods
    if text.endswith("."):
        text = text[:-1]
        
    # Remove common auxiliary verbs at the end of the sentence
    text = re.sub(r'\s+(است|هست|می‌باشد)$', '', text)
    
    return text.strip()

# Read the file "questions.csv" from the data folder
questions = pd.read_csv("data/questions.csv")

answers_list = []

# Generate an answer for each question sequentially with a small delay
for index, row in questions.iterrows():
    question_text = row['question']
    print(f"Question {index + 1}: {question_text}")
    print("Answer: ", end="", flush=True)
    
    try:
        response_text = ""
        
        # Invoke the chain using stream
        for chunk in chain.stream({"question": question_text}):
            # Print each chunk immediately as it arrives
            print(chunk, end="", flush=True)
            response_text += chunk
            
        print("\n" + "-"*40) # Add a newline and separator after the full stream
        
        # Apply post-processing to clean the output for the autograder
        clean_text = clean_response(response_text)
        answers_list.append(clean_text)
        
    except Exception as e:
        print(f"\nError processing row {index}: {e}")
        print("-" * 40)
        # Fallback to the requested default failure state
        answers_list.append("نمی‌دانم")
    
    # Sleep to respect API rate limits
    time.sleep(2)

# Create the submission dataframe
submission = pd.DataFrame({"answer": answers_list})

Question 1: همسر زئوس چه کسی است؟
Answer: 
Error processing row 0: Error calling model 'gemini-flash-lite-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 9h49m51.937300998s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com

In [ ]:
import zipfile
import os

if not os.path.exists(os.path.join(os.getcwd(), 'Olympians.ipynb')):
    %notebook -e Olympians.ipynb

def compress(file_names): 
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

configs = {
    "messages": len(prompt.messages),
    "temperature": model.temperature
}

with open('model_configs.json', 'w') as json_file:
    json.dump(configs, json_file, indent=4)

submission.to_csv('submission.csv', index=False)
pd.DataFrame(examples).to_csv('examples.csv', index=False)

file_names = ['Olympians.ipynb', 'submission.csv',
              'examples.csv','model_configs.json' ]
compress(file_names)

File Paths:
['Olympians.ipynb', 'submission.csv', 'examples.csv', 'model_configs.json']
